In [14]:
import pandas as pd

df = pd.read_csv("leads.csv")
print(df.shape)
df.head(3)

(9160, 20)


,lead_id,created_at,source,city,area,property_type,budget_pkr_lac,bedrooms,first_response_minutes,calls_made,total_call_seconds,whatsapp_replies,site_visits,agent_experience_years,is_overseas,referred_by_existing_client,has_financing_approved,token_amount_received_pkr,crm_record_hash,converted
0,MGC-104067,2025-09-30 19:35:11,Facebook Ads,Rawalpindi,Bahria Town,Farmhouse,801.0,1.0,12.0,2,72.0,4,1,4.7,0,0,1,0.0,8637176927,0
1,MGC-108870,2024-05-25 14:40:03,Instagram,Islamabad,Bahria Town,Commercial Shop,140.0,NaN,97.0,1,57.0,0,1,2.0,0,0,0,0.0,5937285284,0
2,MGC-101529,2024-07-30 02:06:11,Facebook Ads,Faisalabad,DHA,Penthouse,325.0,1.0,38.0,0,0.0,2,0,0.2,1,0,1,0.0,9807118704,0


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9160 entries, 0 to 9159
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   lead_id                      9160 non-null   str    
 1   created_at                   9160 non-null   str    
 2   source                       9160 non-null   str    
 3   city                         9160 non-null   str    
 4   area                         8683 non-null   str    
 5   property_type                9160 non-null   str    
 6   budget_pkr_lac               8876 non-null   float64
 7   bedrooms                     5558 non-null   float64
 8   first_response_minutes       8984 non-null   float64
 9   calls_made                   9160 non-null   int64  
 10  total_call_seconds           9160 non-null   float64
 11  whatsapp_replies             9160 non-null   int64  
 12  site_visits                  9160 non-null   int64  
 13  agent_experience_years       

In [16]:
print(df.isnull().sum())

lead_id                           0
created_at                        0
source                            0
city                              0
area                            477
property_type                     0
budget_pkr_lac                  284
bedrooms                       3602
first_response_minutes          176
calls_made                        0
total_call_seconds                0
whatsapp_replies                  0
site_visits                       0
agent_experience_years          401
is_overseas                       0
referred_by_existing_client       0
has_financing_approved            0
token_amount_received_pkr         0
crm_record_hash                   0
converted                         0
dtype: int64


In [17]:
# Confirm the leakage before dropping it
print(df.groupby("converted")["token_amount_received_pkr"]
        .agg(mean="mean", median="median", pct_nonzero=lambda s: (s > 0).mean()))

                   mean    median  pct_nonzero
converted                                     
0          1.037298e+04       0.0      0.00997
1          1.132114e+06  758500.0      1.00000


In [18]:
# Confirm the duplicate pattern before dropping
dup_hashes = df["crm_record_hash"][df["crm_record_hash"].duplicated(keep=False)]
example_hash = dup_hashes.iloc[0]
df[df["crm_record_hash"] == example_hash]

,lead_id,created_at,source,city,area,property_type,budget_pkr_lac,bedrooms,first_response_minutes,calls_made,total_call_seconds,whatsapp_replies,site_visits,agent_experience_years,is_overseas,referred_by_existing_client,has_financing_approved,token_amount_received_pkr,crm_record_hash,converted
16,MGC-104183,2024-07-03 16:49:50,Property Portal,Islamabad,Blue World City,Plot,29.0,NaN,22.0,1,76.0,3,0,2.2,1,0,0,0.0,4221845300,0
8704,MGC-104183-B,2024-07-03 16:49:50,Property Portal,Islamabad,Blue World City,Plot,29.0,NaN,22.0,1,76.0,3,0,2.2,1,0,0,0.0,4221845300,0


In [19]:
before = len(df)
df = df.drop_duplicates(subset="crm_record_hash", keep="first")
print(f"Dropped {before - len(df)} duplicate leads")

city_mapping = {
    "ISLAMABAD": "Islamabad", "ISB": "Islamabad",
    "RAWALPINDI": "Rawalpindi", "Rwp": "Rawalpindi",
    "LAHORE": "Lahore", "PESHAWAR": "Peshawar", "KARACHI": "Karachi",
    "khi": "Karachi", "MULTAN": "Multan", "FAISALABAD": "Faisalabad",
    "ABBOTTABAD": "Abbottabad", "GUJRANWALA": "Gujranwala",
}
df["city"] = df["city"].replace(city_mapping)
df["area"] = df["area"].fillna("Unknown")

df = df.drop(columns=[
    "lead_id",
    "crm_record_hash",
    "created_at",
    "bedrooms",
    "token_amount_received_pkr",
])

print(df.shape)
df["converted"].value_counts(normalize=True)

Dropped 160 duplicate leads
(9000, 15)


converted
0    0.930444
1    0.069556
Name: proportion, dtype: float64

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["converted"])
y = df["converted"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

((7200, 14), (1800, 14))

In [22]:
from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression



numeric_cols = X_train.select_dtypes(include=["float64", "int64"]).columns.tolist()

categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()



numeric_pipe = Pipeline([

    ("impute", SimpleImputer(strategy="median")),

    ("scale", StandardScaler()),

])

categorical_pipe = Pipeline([

    ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),

    ("ohe", OneHotEncoder(handle_unknown="ignore")),

])



preprocessor = ColumnTransformer([

    ("num", numeric_pipe, numeric_cols),

    ("cat", categorical_pipe, categorical_cols),

])



model = Pipeline([

    ("preprocessor", preprocessor),

    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),

])



model.fit(X_train, y_train);


C:\Users\hp\AppData\Local\Temp\ipykernel_17316\1100428778.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()


In [23]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix, classification_report

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

dummy = DummyClassifier(strategy="most_frequent", random_state=42).fit(X_train, y_train)
dummy_proba = dummy.predict_proba(X_test)[:, 1]

print("Majority-class baseline PR-AUC:", round(average_precision_score(y_test, dummy_proba), 4))
print("Logistic Regression   PR-AUC:", round(average_precision_score(y_test, y_proba), 4))
print()
print("Logistic Regression  ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))

Majority-class baseline PR-AUC: 0.0694
Logistic Regression   PR-AUC: 0.3635

Logistic Regression  ROC-AUC: 0.838


In [24]:
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, digits=3))

[[1245  430]
 [  28   97]]

              precision    recall  f1-score   support

           0      0.978     0.743     0.845      1675
           1      0.184     0.776     0.298       125

    accuracy                          0.746      1800
   macro avg      0.581     0.760     0.571      1800
weighted avg      0.923     0.746     0.807      1800



In [10]:
import pickle
with open("lead_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("Saved lead_model.pkl")

Saved lead_model.pkl
